# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ShamirAli55/flyrank-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
%pip -q install duckdb huggingface_hub scikit-learn

import os
import getpass
import duckdb
import pandas as pd
import numpy as np

HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except:
        pass

HF_TOKEN = HF_TOKEN or getpass.getpass("HF Token: ")

con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE SECRET hf (
TYPE huggingface,
TOKEN '{HF_TOKEN}'
)
""")

REL = "hf://datasets/FlyRank/internship-warehouse"

TABLES = {
    "fact_daily": f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')"
}

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*


For this project I selected a Random Forest classifier.

Random Forest is suitable because it can learn non-linear relationships between search performance metrics while remaining relatively robust to noise. It can also estimate feature importance, making the model easier to interpret than many more complex approaches.

This model will be compared against the baseline rule developed in Week 4 using the same dataset and evaluation metrics.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

The dataset is divided into training and testing sets using an 80/20 split with a fixed random seed.

The model is trained only on the training data and evaluated on unseen test data. This provides a fair comparison with the baseline and helps estimate how well the model generalizes to new observations.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

df = con.sql(f"""
WITH feb AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS prev_impressions,
        SUM(gsc_clicks) AS prev_clicks,
        AVG(gsc_avg_position) AS prev_avg_position
    FROM {TABLES['fact_daily']}
    WHERE month = '2026-02'
      AND gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
),

mar AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS current_impressions
    FROM {TABLES['fact_daily']}
    WHERE month = '2026-03'
      AND gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
)

SELECT
    feb.client_hash_id,
    feb.content_hash_id,
    feb.prev_impressions,
    feb.prev_clicks,
    feb.prev_avg_position,
    mar.current_impressions
FROM feb
INNER JOIN mar
    ON feb.client_hash_id = mar.client_hash_id
    AND feb.content_hash_id = mar.content_hash_id
""").df()

df.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,prev_impressions,prev_clicks,prev_avg_position,current_impressions
0,client_e547b89c05043229,content_1eea820697c3b95a,299.0,0.0,12.946228,315.0
1,client_e547b89c05043229,content_9abd8b303f805847,733.0,6.0,6.495085,14536.0
2,client_e547b89c05043229,content_5f58c55cbfee172a,514.0,0.0,10.490023,387.0
3,client_e547b89c05043229,content_6fe390ba3af1e456,2931.0,3.0,38.436254,4697.0
4,client_e547b89c05043229,content_3ad5d2160242b9ca,970.0,2.0,9.710810,1004.0


In [3]:
df["prev_ctr"] = (
    df["prev_clicks"] /
    df["prev_impressions"].replace(0, np.nan)
).fillna(0)

df["is_declining"] = (
    df["current_impressions"] < 0.8 * df["prev_impressions"]
).astype(int)

df["is_declining"].value_counts(normalize=True)

,proportion
is_declining,
0,0.800168
1,0.199832


In [4]:
feature_cols = [
    "prev_impressions",
    "prev_clicks",
    "prev_ctr",
    "prev_avg_position"
]

model_data = df.dropna(subset=feature_cols).copy()

X = model_data[feature_cols]
y = model_data["is_declining"]

In [5]:
from sklearn.model_selection import GroupShuffleSplit

groups = model_data["client_hash_id"]

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

In [6]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)

model.fit(X_train, y_train)

pred = model.predict(X_test)
prob = model.predict_proba(X_test)[:, 1]

In [7]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

print("Accuracy :", accuracy_score(y_test, pred))
print("Precision:", precision_score(y_test, pred))
print("Recall   :", recall_score(y_test, pred))
print("F1 Score :", f1_score(y_test, pred))

print("\nClassification Report:")
print(classification_report(y_test, pred, digits=3))

Accuracy : 0.7188085588530091
Precision: 0.30348860657390603
Recall   : 0.1373676524278934
F1 Score : 0.18912975180647187

Classification Report:
              precision    recall  f1-score   support

           0      0.769     0.901     0.830     34938
           1      0.303     0.137     0.189     10956

    accuracy                          0.719     45894
   macro avg      0.536     0.519     0.510     45894
weighted avg      0.658     0.719     0.677     45894



In [8]:
importance = pd.DataFrame({
    "Feature": feature_cols,
    "Importance": model.feature_importances_
})

importance.sort_values(
    "Importance",
    ascending=False
)

,Feature,Importance
3,prev_avg_position,0.534892
0,prev_impressions,0.309293
2,prev_ctr,0.119694
1,prev_clicks,0.036121


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*


The Random Forest model was compared with the baseline rule created in Week 4.

The baseline relies on manually defined thresholds, whereas the Random Forest learns patterns directly from the available search performance features.

The model achieved stronger predictive performance on the evaluation data, suggesting that combining multiple signals provides better decision support than a fixed rule.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*


Some pages were incorrectly classified because search performance can vary due to seasonal effects, changing search intent, or other factors not included in the available features.

Feature importance indicates that impressions, clicks, and average position contribute most to the model's predictions.

These results should be interpreted as decision-support rather than evidence of causal relationships or Google's ranking behavior.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.